# LeetCode #1473: Paint House III

https://leetcode.com/problems/paint-house-iii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m^n \cdot n)$ | $O(n)$ |
| **Optimal: 3D DP ★** | $O(n \cdot target \cdot m^2)$ | $O(n \cdot target \cdot m)$ |

---

## Understanding the Methods

### Brute Force
Try every color assignment for all uncolored houses, count neighborhoods, and keep the minimum cost if exactly `target` neighborhoods are formed. Exponential — impractical beyond tiny inputs.

### Optimal: 3D DP ★
`dp[i][j][k]` = minimum cost to paint houses `0..i` such that house `i` has color `j` and there are exactly `k` neighborhoods so far. Transition: for each previous color `prevJ`, if `prevJ != j` then neighborhoods increment. Already-colored houses have zero cost and force a specific color. Answer: `min over j of dp[n-1][j][target]`.

**Why this is better than Brute Force:** The DP table has $O(n \cdot target \cdot m)$ states with $O(m)$ transitions each — polynomial versus exponential.

**Constraints:**
* $1 \leq m \leq 20$ (number of colors)
* $1 \leq n \leq 100$ (number of houses)
* $1 \leq target \leq n$
* $0 \leq$ `houses[i]` $\leq m$
* `cost[i][j]` $\geq 0$

## Solutions

### C#

In [ ]:
public class Solution {
    public int MinCost(int[] houses, int[][] cost, int m, int n, int target) {
        const int INF = int.MaxValue / 2;
        // dp[i][j][k] = min cost to paint houses 0..i, last color = j+1, k neighborhoods
        var dp = new int[n, m, target + 1];
        for (int i = 0; i < n; i++)
            for (int j = 0; j < m; j++)
                for (int k = 0; k <= target; k++)
                    dp[i, j, k] = INF;

        // Initialise first house
        if (houses[0] != 0) {
            dp[0, houses[0] - 1, 1] = 0; // Already painted — no cost, 1 neighborhood
        } else {
            for (int j = 0; j < m; j++)
                dp[0, j, 1] = cost[0][j]; // Paint it any color — 1 neighborhood
        }

        for (int i = 1; i < n; i++) {
            for (int j = 0; j < m; j++) {
                // Skip if this house is already painted with a different color
                if (houses[i] != 0 && houses[i] - 1 != j) continue;
                int paintCost = houses[i] == 0 ? cost[i][j] : 0;
                for (int k = 1; k <= target; k++) {
                    for (int pj = 0; pj < m; pj++) {
                        if (dp[i - 1, pj, k - (pj != j ? 1 : 0)] == INF) continue;
                        int prev = (pj != j) ? (k >= 2 ? dp[i - 1, pj, k - 1] : INF) : dp[i - 1, pj, k];
                        if (prev == INF) continue;
                        dp[i, j, k] = Math.Min(dp[i, j, k], prev + paintCost);
                    }
                }
            }
        }

        int ans = INF;
        for (int j = 0; j < m; j++)
            ans = Math.Min(ans, dp[n - 1, j, target]);
        return ans == INF ? -1 : ans;
    }
}

### Python

In [ ]:
from typing import List

class Solution:
    def min_cost(self, houses: List[int], cost: List[List[int]], m: int, n: int, target: int) -> int:
        INF = float('inf')
        # dp[i][j][k] = min cost to paint houses 0..i, last color = j+1, k neighborhoods
        dp = [[[INF] * (target + 1) for _ in range(m)] for _ in range(n)]

        # Initialise first house
        if houses[0] != 0:
            dp[0][houses[0] - 1][1] = 0  # Already painted — no cost, 1 neighborhood
        else:
            for j in range(m):
                dp[0][j][1] = cost[0][j]  # Paint it any color — 1 neighborhood

        for i in range(1, n):
            for j in range(m):
                if houses[i] != 0 and houses[i] - 1 != j:
                    continue  # Skip: house pre-painted with different color
                paint_cost = 0 if houses[i] != 0 else cost[i][j]
                for k in range(1, target + 1):
                    for pj in range(m):
                        # Determine previous neighborhood count
                        prev_k = k - 1 if pj != j else k
                        if prev_k < 1 or dp[i - 1][pj][prev_k] == INF:
                            continue
                        dp[i][j][k] = min(dp[i][j][k], dp[i - 1][pj][prev_k] + paint_cost)

        ans = min(dp[n - 1][j][target] for j in range(m))
        return ans if ans < INF else -1

### Go

In [ ]:
func minCost(houses []int, cost [][]int, m int, n int, target int) int {
    const INF = 1 << 30
    // dp[i][j][k] = min cost to paint houses 0..i, last color = j+1, k neighborhoods
    dp := make([][][]int, n)
    for i := range dp {
        dp[i] = make([][]int, m)
        for j := range dp[i] { dp[i][j] = make([]int, target+1); for k := range dp[i][j] { dp[i][j][k] = INF } }
    }
    // Initialise first house
    if houses[0] != 0 {
        dp[0][houses[0]-1][1] = 0
    } else {
        for j := 0; j < m; j++ { dp[0][j][1] = cost[0][j] }
    }
    for i := 1; i < n; i++ {
        for j := 0; j < m; j++ {
            if houses[i] != 0 && houses[i]-1 != j { continue }
            paintCost := 0
            if houses[i] == 0 { paintCost = cost[i][j] }
            for k := 1; k <= target; k++ {
                for pj := 0; pj < m; pj++ {
                    prevK := k
                    if pj != j { prevK = k - 1 }
                    if prevK < 1 || dp[i-1][pj][prevK] == INF { continue }
                    if v := dp[i-1][pj][prevK] + paintCost; v < dp[i][j][k] { dp[i][j][k] = v }
                }
            }
        }
    }
    ans := INF
    for j := 0; j < m; j++ { if dp[n-1][j][target] < ans { ans = dp[n-1][j][target] } }
    if ans == INF { return -1 }
    return ans
}

### Rust

In [ ]:
impl Solution {
    pub fn min_cost(houses: Vec<i32>, cost: Vec<Vec<i32>>, m: i32, n: i32, target: i32) -> i32 {
        let (m, n, target) = (m as usize, n as usize, target as usize);
        const INF: i32 = i32::MAX / 2;
        // dp[i][j][k] = min cost to paint houses 0..i, last color = j+1, k neighborhoods
        let mut dp = vec![vec![vec![INF; target + 1]; m]; n];

        // Initialise first house
        let h0 = houses[0] as usize;
        if h0 != 0 {
            dp[0][h0 - 1][1] = 0;
        } else {
            for j in 0..m { dp[0][j][1] = cost[0][j]; }
        }

        for i in 1..n {
            let hi = houses[i] as usize;
            for j in 0..m {
                if hi != 0 && hi - 1 != j { continue; }
                let paint_cost = if hi == 0 { cost[i][j] } else { 0 };
                for k in 1..=target {
                    for pj in 0..m {
                        let prev_k = if pj != j { k.wrapping_sub(1) } else { k };
                        if prev_k == 0 || prev_k > target { continue; }
                        if dp[i - 1][pj][prev_k] == INF { continue; }
                        let val = dp[i - 1][pj][prev_k] + paint_cost;
                        if val < dp[i][j][k] { dp[i][j][k] = val; }
                    }
                }
            }
        }

        let ans = (0..m).map(|j| dp[n - 1][j][target]).min().unwrap();
        if ans == INF { -1 } else { ans }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `houses=[0,0,0,0,0], cost=[[1,10],[10,1],[10,1],[1,10],[5,1]], m=2, n=5, target=3`
Optimal coloring creates exactly 3 neighborhoods at minimum cost. Answer: **9**.

### 2. Slightly Complex
**Input:** `houses=[0,2,1,2,0], cost=[[1,10],[10,1],[10,1],[1,10],[5,1]], m=2, n=5, target=3`
Houses 2, 4, 6 are pre-painted (1-indexed). The DP only considers valid color assignments for unpainted houses. Answer: **11**.

### 3. Edge Case: Time Factor
**Input:** `n=100, m=20, target=100`
The DP table has $100 \times 20 \times 100 = 200{,}000$ states, each iterating 20 previous colors — $4 \times 10^6$ operations. Maximum complexity.

### 4. Edge Case: Space Factor
**Input:** `n=100, m=20, target=100`
The full $100 \times 20 \times 101$ DP table uses $\approx 200{,}000$ integers. At 4 bytes each, $\approx 800$ KB — largest allocation for this problem.

### 5. Almost-Impossible but Plausible
**Input:** `houses=[1,2,1,1,1], cost=[[1,1],[1,1],[1,1],[1,1],[1,1]], m=2, n=5, target=1`
All pre-painted houses mix colors 1 and 2. To form exactly 1 neighborhood, all houses must share the same color — but house 2 is color 2 while houses 1,3,4,5 are color 1. Impossible. Answer: **-1**.